# Qwen3.6-35B-A3B → 64-layer looped transformer (Colab G4 · RTX PRO 6000 Blackwell)
All logic lives in the repo; every stage runs as a fresh subprocess (`python -m qloop …`) with its own
run dir, log file and crash report. On failure the cell prints the **hints + traceback**; re-run with
`--debug 2` (NaN hooks, C++ traces) or `--debug 3` (CUDA_LAUNCH_BLOCKING, anomaly mode, memory timeline).
Runtime: *Runtime → Change runtime type → G4*. Secrets: `HF_TOKEN` (required), optional `WANDB_API_KEY`, `TEACHER_API_KEY`, `DAYTONA_API_KEY`.

In [ ]:
REPO, BRANCH = "https://github.com/BaiTian6641/Qwen3.6-35B-A3B-Loop60.git", "main"
RUN_ROOT = "/content/drive/MyDrive/qloop_runs"   # checkpoints survive session resets
from google.colab import drive; drive.mount("/content/drive")
!test -d /content/qloop || git clone -b $BRANCH $REPO /content/qloop
%cd /content/qloop
!git pull --ff-only && bash scripts/colab_setup.sh

In [ ]:
from qloop.colab import setup_env, q, tail_metrics
setup_env(run_root=RUN_ROOT, hf_home="/content/hf_cache")
q("doctor --freeze requirements/colab.lock.txt --model Qwen/Qwen3.6-35B-A3B")

### S0 · probes (backward through frozen base, real-model surgery check, 64-layer core-range ranking) + ESFT scan

In [ ]:
q("probe -c configs/stages/s0_probe.yaml --backward --surgery --core-ranges --n 8 --seq-len 4096")
q("scan-experts -c configs/stages/s0_probe.yaml --batches 256 --seq-len 4096")

### S1 → S4 · run a stage (re-running a cell resumes from the last checkpoint)

In [ ]:
STAGE = "s1_loop_heal"   # s1_loop_heal | s2_sft | s3_long_lsa | s4_rl
q(f"train -c configs/stages/{STAGE}.yaml")
tail_metrics(STAGE)

### Evaluation (loop-depth sweep) · optional OpenAI-compatible server for Harbor / tau-bench

In [ ]:
q("eval -c configs/eval/suite.yaml --ckpt " + RUN_ROOT + "/s4_rl")
# q("serve -c configs/eval/suite.yaml --ckpt " + RUN_ROOT + "/s4_rl --port 8000")  # then: q("external harbor -c configs/eval/suite.yaml")